# Lab 2 — AI Engineer spikes

Два коротких эксперимента для рискованных допущений AI-pipeline. Ноутбук не вызывает Mistral API и не требует секретов.

## Spike 1 — compact structured output

Проверяем управляемую часть нагрузки: размер схемы и типичного ответа для rich-формата `title+rationale+keywords` и compact `titles[]`. Это **не измерение latency провайдера**, а измерение размера structured payload.

In [1]:
import json

rich_schema = {
    'type': 'object', 'properties': {'teachers': {'type': 'array', 'items': {
        'type': 'object', 'properties': {'teacher_id': {'type': 'integer'}, 'topics': {
            'type': 'array', 'items': {'type': 'object', 'properties': {
                'title': {'type': 'string'}, 'rationale': {'type': 'string'},
                'keywords': {'type': 'array', 'items': {'type': 'string'}}},
                'required': ['title','rationale','keywords'], 'additionalProperties': False}},
        'required': ['teacher_id','topics'], 'additionalProperties': False}}},
    'required': ['teachers'], 'additionalProperties': False}
}
compact_schema = {
    'type': 'object', 'properties': {'teachers': {'type': 'array', 'items': {
        'type': 'object', 'properties': {'teacher_id': {'type': 'integer'},
        'titles': {'type': 'array', 'items': {'type': 'string'}}},
        'required': ['teacher_id','titles'], 'additionalProperties': False}}},
    'required': ['teachers'], 'additionalProperties': False
}
titles = [
 'Разработка информационной системы управления учебными практиками с распределением студентов и контролем сроков',
 'Разработка веб-сервиса мониторинга оборудования с выявлением аномалий и формированием уведомлений',
 'Разработка системы анализа научных публикаций с тематической классификацией и поиском связанных работ',
 'Разработка интеллектуального помощника для подготовки технических заданий с проверкой полноты требований',
 'Разработка системы компьютерного зрения для контроля качества продукции с обнаружением дефектов',
 'Разработка платформы управления проектами с прогнозированием сроков и анализом загрузки команды',
 'Разработка программного комплекса анализа телеметрии с диагностикой отклонений и визуализацией показателей'
]
rich = {'teachers':[{'teacher_id':1,'topics':[{'title':t,
  'rationale':'Реализовать backend/API, обработку данных, интерфейс и оценить качество ключевой функции.',
  'keywords':['API','данные','анализ','интерфейс']} for t in titles]}]}
compact = {'teachers':[{'teacher_id':1,'titles':titles}]}
b = lambda obj: len(json.dumps(obj, ensure_ascii=False, separators=(',',':')).encode('utf-8'))
print('rich schema:', b(rich_schema), 'bytes')
print('compact schema:', b(compact_schema), 'bytes')
print('rich response:', b(rich), 'bytes')
print('compact response:', b(compact), 'bytes')
print('compact/rich ratio:', round(b(compact)/b(rich), 2))


rich schema: 500 bytes
compact schema: 302 bytes
rich response: 3157 bytes
compact response: 1407 bytes
compact/rich ratio: 0.44


**Вывод:** compact response в этом представительном примере занимает около 44% rich-варианта. Поэтому rationale/keywords не должны генерироваться в первом AI-вызове, если downstream их не требует.

## Spike 2 — lexical fallback

Проверяем, можно ли использовать один порог 45 для lexical-only режима. Код импортирует текущую production-функцию из `backend/app/services/similarity.py`.

In [2]:
from pathlib import Path
import sys

candidates = [Path.cwd() / 'backend', Path.cwd().parent / 'backend']
backend = next((p for p in candidates if p.exists()), None)
if backend is None:
    raise RuntimeError('Запустите notebook из корня репозитория или из папки notebooks/')
sys.path.insert(0, str(backend))

from app.services.similarity import lexical_fallback_similarity

pairs = [
 ('Многоагентная система глубинного изучения математических дисциплин с автогенерацией пермонализированных заданий',
  'Разработка системы автоматической генерации и проверки учебных заданий с адаптацией сложности'),
 ('Многоагентная система фиксации и управления историей разработки учебных и научных работ',
  'Разработка системы управления и анализа истории разработки научных проектов вуза с визуализацией зависимостей'),
 ('Разработка системы обнаружения объектов на строительных объектах по изображениям с формированием визуальных протоколов',
  'Разработка системы оценки качества строительных швов с использованием компьютерного зрения и сегментацией дефектов'),
]
scores = [lexical_fallback_similarity(a,b) for a,b in pairs]
for i, score in enumerate(scores, 1): print(f'pair {i}: {score}')
print('flagged with threshold 45:', f'{sum(s >= 45 for s in scores)}/3')
print('flagged with threshold 25:', f'{sum(s >= 25 for s in scores)}/3')


pair 1: 30.0
pair 2: 37.3
pair 3: 33.7
flagged with threshold 45: 0/3
flagged with threshold 25: 3/3


**Вывод:** общий threshold 45 пропускает все три вручную отмеченные близкие пары. Для lexical fallback нужен отдельный более низкий порог (в текущем pipeline — 25/50), а основным режимом остаются локальные multilingual embeddings.